# Daily Embedding Classifier — Home Ventilator Data

**Goal**: Compress each patient-day into a 32-dim embedding using a shared Transformer,
then classify label 0 vs label 1 using XGBoost / Random Forest per day.

**Pipeline**:
```
Raw readings (per 24k-row chunk per patient per unified_day)
  → Shared Transformer Encoder  (BCE loss, patient-level label)
  → 32-dim chunk embedding
  → Average chunks → 1 embedding per (patient, unified_day)
  → XGBoost / RF on 32-dim features → label 0 or 1 per row
  → Results table: rows=patients, columns=days
```

**Data**: 86 patients (patient 68 dropped), ~1/3 label 1, 7 days of ventilator readings
- unified_day 6 = earliest day (furthest from event)
- unified_day 0 = latest day  (closest to event)
- Each (patient, day) row is one independent prediction


---
## Step 1 — Install & Import

In [2]:
!pip install -q xgboost scikit-learn

In [1]:
import os
import gc
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.cuda.amp import GradScaler, autocast
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, classification_report,
                              confusion_matrix)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.decomposition import PCA
from collections import Counter, defaultdict
from matplotlib.colors import LinearSegmentedColormap
import xgboost as xgb
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'PyTorch: {torch.__version__}')


Device : cuda
PyTorch: 2.10.0+cu128


---
## Step 2 — Mount Drive & Load Data

In [3]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/My Drive/Monmouth_University/2025_RA/Medical_AI/Cleaned_Data/last7days_rawdata_label0&1_parquet"

LOAD_COLS = [
    'patient_id', 'label', 'timestamp', 'end_date',
    'countdown_days_target', 'test_type', 'test_value_scaled',
    'hour_of_day_scaled', 'collection'
]

df_train = pd.read_parquet(os.path.join(ROOT, 'train.parquet'), columns=LOAD_COLS)
df_val   = pd.read_parquet(os.path.join(ROOT, 'val.parquet'),   columns=LOAD_COLS)
df_test  = pd.read_parquet(os.path.join(ROOT, 'test.parquet'),  columns=LOAD_COLS)

for df in [df_train, df_val, df_test]:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['end_date']  = pd.to_datetime(df['end_date'],  errors='coerce')

print(f'Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}')
print(f'Columns: {df_train.columns.tolist()}')
for name, df in [('train',df_train),('val',df_val),('test',df_test)]:
    print(f'  {name}: {df.memory_usage(deep=True).sum()/1024**3:.2f} GB')
df_train.head(3)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: 73,041,029  |  Val: 20,671,364  |  Test: 26,861,042
Columns: ['patient_id', 'label', 'timestamp', 'end_date', 'countdown_days_target', 'test_type', 'test_value_scaled', 'hour_of_day_scaled', 'collection']
  train: 10.95 GB
  val: 3.06 GB
  test: 4.00 GB


,patient_id,label,timestamp,end_date,countdown_days_target,test_type,test_value_scaled,hour_of_day_scaled,collection
0,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,6,pressure,0.027914,0.541667,train
1,2,0,2023-10-25 13:04:44.200,2023-10-31 22:17:08,6,flow,0.486997,0.541667,train
2,2,0,2023-10-25 13:04:44.400,2023-10-31 22:17:08,6,flow,0.002517,0.541667,train


---
## Step 3 — Fix Countdown Days, Drop Bad Patients, Trim Columns

**unified_day (0..6, 7 days)**:
- Label 1: use `countdown_days_target` directly (0..6), no dropping, no shifting
  - unified_day 0 = day of hospital admission (sparse but kept)
  - unified_day 6 = earliest day (6 days before hospital)
- Label 0: calculate days back from `end_date`
  - unified_day 6 = last day of monitoring window
  - unified_day 0 = first day of monitoring window

**Drop**: patient 68 — only 3 days of data.

**Trim**: keep only columns needed for training to save RAM.


In [4]:
# ── Drop insufficient patients ───────────────────────────────────────────────
PATIENTS_TO_DROP = [68]
for df in [df_train, df_val, df_test]:
    mask = df['patient_id'].isin(PATIENTS_TO_DROP)
    df.drop(df[mask].index, inplace=True)
print(f'Dropped patients: {PATIENTS_TO_DROP}')

# ── Build unified_day (0..6) ──────────────────────────────────────────────────
def add_unified_day(df):
    df['unified_day'] = -1
    mask1 = df['label'] == 1
    df.loc[mask1, 'unified_day'] = (
        df.loc[mask1, 'countdown_days_target'].clip(0, 6)).astype(int)
    mask0 = df['label'] == 0
    days_from_end = (
        df.loc[mask0, 'end_date'].dt.normalize()
      - df.loc[mask0, 'timestamp'].dt.normalize()
    ).dt.days
    df.loc[mask0, 'unified_day'] = (6 - days_from_end).clip(0, 6).astype(int)
    drop_idx = df[~df['unified_day'].between(0, 6)].index
    df.drop(drop_idx, inplace=True)
    df['unified_day'] = df['unified_day'].astype(int)

print('\nAdding unified_day...')
add_unified_day(df_train); gc.collect()
add_unified_day(df_val);   gc.collect()
add_unified_day(df_test);  gc.collect()

# ── Compute time_from_start_scaled (fit on train only) ───────────────────────
from sklearn.preprocessing import MinMaxScaler
print('\nComputing time_from_start_scaled...')
for df in [df_train, df_val, df_test]:
    patient_min_ts = df.groupby('patient_id')['timestamp'].transform('min')
    df['time_from_start_seconds'] = (df['timestamp'] - patient_min_ts).dt.total_seconds()

tfs_scaler = MinMaxScaler()
df_train['time_from_start_scaled'] = tfs_scaler.fit_transform(df_train[['time_from_start_seconds']])
df_val['time_from_start_scaled']   = tfs_scaler.transform(df_val[['time_from_start_seconds']])
df_test['time_from_start_scaled']  = tfs_scaler.transform(df_test[['time_from_start_seconds']])

# ── Keep only columns needed — drop timestamp/end_date/etc immediately ────────
KEEP_COLS = ['patient_id', 'label', 'unified_day',
             'test_type', 'test_value_scaled',
             'hour_of_day_scaled', 'time_from_start_scaled', 'collection']

for df, name in [(df_train,'train'), (df_val,'val'), (df_test,'test')]:
    drop_cols = [c for c in df.columns if c not in KEEP_COLS]
    df.drop(columns=drop_cols, inplace=True)
    df.reset_index(drop=True, inplace=True)
    gc.collect()
    print(f'{name:>6}: {len(df):,} rows  {df.memory_usage(deep=True).sum()/1024**3:.2f} GB')

all_df = pd.concat([df_train, df_val, df_test], ignore_index=True)
print('\n=== unified_day distribution by label ===')
print(all_df.groupby(['label','unified_day']).size().rename('rows').reset_index().to_string(index=False))
print(f'\nTotal patients: {all_df["patient_id"].nunique()}')
print(f'Label 0: {(all_df.drop_duplicates("patient_id")["label"]==0).sum()}  '
      f'Label 1: {(all_df.drop_duplicates("patient_id")["label"]==1).sum()}')
del all_df; gc.collect()


Dropped patients: [68]

Adding unified_day...

Computing time_from_start_scaled...
 train: 71,871,971 rows  10.51 GB
   val: 20,671,364 rows  2.98 GB
  test: 26,861,042 rows  3.90 GB

=== unified_day distribution by label ===
 label  unified_day     rows
     0            0 15609482
     0            1 10484170
     0            2 10331909
     0            3  8405382
     0            4 10450702
     0            5  9457101
     0            6  8723024
     1            0  7683053
     1            1  6181350
     1            2  6634248
     1            3  6374367
     1            4  7176347
     1            5  6199971
     1            6  5693271

Total patients: 85
Label 0: 57  Label 1: 28


0

---
## Step 4 — Label Encoder for test_type

In [6]:
uniq_types   = np.union1d(df_train['test_type'].unique(),
                            df_val['test_type'].unique())
le_test_type = LabelEncoder().fit(uniq_types)

NUM_REAL_TYPES = len(le_test_type.classes_)
PAD_TYPE_ID    = NUM_REAL_TYPES
VOCAB_SIZE     = NUM_REAL_TYPES + 1

print(f'test_type classes : {le_test_type.classes_}')
print(f'PAD_TYPE_ID       : {PAD_TYPE_ID}')
print(f'VOCAB_SIZE         : {VOCAB_SIZE}')


test_type classes : ['flow' 'pressure']
PAD_TYPE_ID       : 2
VOCAB_SIZE         : 3


---
## Step 5 — Dataset: 24,000-row chunks

Groups by `(patient_id, unified_day)`. Each day split into 24,000-row chunks
(~40 min, same as regression model). Each chunk = one training sample.
At embedding extraction: average all chunks per patient-day → 1 embedding per day.

`BATCH_SIZE=32'


In [7]:
class PatientDayDataset(Dataset):
    """
    Groups by (patient_id, unified_day), splits into chunk_size-row chunks.
    Uses numpy arrays for fast __getitem__ instead of slow pandas .loc
    """
    def __init__(self, df, le_test_type, chunk_size=6000):
        self.le      = le_test_type
        self.samples = []

        # Pre-extract numpy arrays — avoids slow pandas .loc per batch
        raw_type_arr = df['test_type'].to_numpy()
        type_id_arr  = np.full(len(raw_type_arr), PAD_TYPE_ID, dtype=np.int64)
        known_mask   = np.isin(raw_type_arr, le_test_type.classes_)
        if known_mask.any():
            type_id_arr[known_mask] = le_test_type.transform(raw_type_arr[known_mask])

        self.type_arr  = type_id_arr
        self.value_arr = df['test_value_scaled'].to_numpy(np.float32)
        self.hour_arr  = df['hour_of_day_scaled'].to_numpy(np.float32)
        self.tfs_arr   = df['time_from_start_scaled'].to_numpy(np.float32)

        df_idx = df.reset_index(drop=True)
        for (pid, uday), sub in df_idx.groupby(['patient_id','unified_day'], sort=False):
            label = int(sub['label'].iloc[0])
            idx   = sub.index.to_numpy()
            for chunk_id, start in enumerate(range(0, len(idx), chunk_size)):
                chunk_idx = idx[start: start + chunk_size]
                if len(chunk_idx) == 0: continue
                self.samples.append((pid, int(uday), label, chunk_id, chunk_idx))

        n_patients   = df['patient_id'].nunique()
        label_counts = Counter(s[2] for s in self.samples)
        cday_counts  = Counter(s[1] for s in self.samples)
        print(f'  Built: {len(self.samples)} chunks from {n_patients} patients')
        print(f'  Label 0 chunks: {label_counts[0]}  |  Label 1 chunks: {label_counts[1]}')
        print(f'  Chunks per unified_day:')
        for d in range(7):
            print(f'    day {d}: {cday_counts.get(d, 0)} chunks')

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        pid, uday, label, chunk_id, idx = self.samples[i]
        return {
            'patient_id':      pid,
            'unified_day':     uday,
            'chunk_id':        chunk_id,
            'label':           torch.tensor(label, dtype=torch.float32),
            'test_type':       torch.from_numpy(self.type_arr[idx]),
            'test_value':      torch.from_numpy(self.value_arr[idx]),
            'hour':            torch.from_numpy(self.hour_arr[idx]),
            'time_from_start': torch.from_numpy(self.tfs_arr[idx]),
        }


def collate_patient_day(batch):
    tt  = [s['test_type']       for s in batch]
    tv  = [s['test_value']      for s in batch]
    ho  = [s['hour']            for s in batch]
    tfs = [s['time_from_start'] for s in batch]

    test_type       = pad_sequence(tt,  batch_first=True, padding_value=PAD_TYPE_ID).long()
    test_value      = pad_sequence(tv,  batch_first=True, padding_value=0.0).float()
    hour            = pad_sequence(ho,  batch_first=True, padding_value=0.0).float()
    time_from_start = pad_sequence(tfs, batch_first=True, padding_value=0.0).float()

    lengths      = torch.tensor([len(t) for t in tt], dtype=torch.long)
    T            = test_type.size(1)
    padding_mask = ~(torch.arange(T).unsqueeze(0) < lengths.unsqueeze(1))

    return {
        'test_type':       test_type,
        'test_value':      test_value,
        'hour':            hour,
        'time_from_start': time_from_start,
        'padding_mask':    padding_mask,
        'label':           torch.stack([s['label']   for s in batch]),
        'patient_id':      [s['patient_id']           for s in batch],
        'unified_day':     [s['unified_day']          for s in batch],
        'chunk_id':        [s['chunk_id']             for s in batch],
    }


from tqdm.auto import tqdm
import time

CHUNK_SIZE = 6000
BATCH_SIZE = 32

# Reset index so numpy positional indexing is correct
df_train.reset_index(drop=True, inplace=True)
df_val.reset_index(drop=True, inplace=True)
df_test.reset_index(drop=True, inplace=True)

print('Building datasets (numpy fast indexing, chunk_size=6000)...')
train_ds = PatientDayDataset(df_train, le_test_type, chunk_size=CHUNK_SIZE)
print()
val_ds   = PatientDayDataset(df_val,   le_test_type, chunk_size=CHUNK_SIZE)
print()
test_ds  = PatientDayDataset(df_test,  le_test_type, chunk_size=CHUNK_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_patient_day, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_patient_day, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_patient_day, num_workers=0, pin_memory=True)

print(f'\n  train batches: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}')

# Speed test
print('\nData loading speed test (3 batches)...')
t0 = time.time()
for i, b in enumerate(train_loader):
    if i == 2: break
elapsed = time.time() - t0
print(f'  3 batches in {elapsed:.2f}s  batch shape: {b["test_type"].shape}')
print('  ✅ Fast!' if elapsed < 5 else '  ⚠️  Still slow')

print('\n=== Label distribution per split ===')
for name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    labels = [s[2] for s in ds.samples]
    n0, n1 = labels.count(0), labels.count(1)
    print(f'  {name:>6}: total={len(labels):>5}  label 0={n0:>4}  label 1={n1:>4}  ratio={n0/max(n1,1):.2f}')


Building datasets (numpy fast indexing, chunk_size=6000)...
  Built: 12143 chunks from 48 patients
  Label 0 chunks: 6871  |  Label 1 chunks: 5272
  Chunks per unified_day:
    day 0: 2471 chunks
    day 1: 1644 chunks
    day 2: 1803 chunks
    day 3: 1482 chunks
    day 4: 1760 chunks
    day 5: 1610 chunks
    day 6: 1373 chunks

  Built: 3493 chunks from 15 patients
  Label 0 chunks: 2015  |  Label 1 chunks: 1478
  Chunks per unified_day:
    day 0: 591 chunks
    day 1: 522 chunks
    day 2: 495 chunks
    day 3: 483 chunks
    day 4: 600 chunks
    day 5: 420 chunks
    day 6: 382 chunks

  Built: 4555 chunks from 22 patients
  Label 0 chunks: 3561  |  Label 1 chunks: 994
  Chunks per unified_day:
    day 0: 860 chunks
    day 1: 654 chunks
    day 2: 570 chunks
    day 3: 536 chunks
    day 4: 622 chunks
    day 5: 624 chunks
    day 6: 689 chunks

  train batches: 380  val: 110  test: 143

Data loading speed test (3 batches)...
  3 batches in 0.27s  batch shape: torch.Size([32,

---
## Step 6 — Model: Shared Transformer Encoder → 32-dim Embedding

- type_emb + value_fc + **Time2Vec(hour)** + **Time2Vec(time_from_start)** → fused token [B, T, 32]
- `Time2Vec` (Kazemi et al., 2019): one linear trend term + learnable sinusoids — captures both trend and periodicity in continuous time signals
- Positional encoding (max_len=30000)
- Transformer encoder (2 layers, 4 heads, 128 FFN dim)
- Mean-pool over valid tokens → 32-dim embedding
- Linear classifier head (BCE training only, discarded at inference)


In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=30000):
        super().__init__()
        pe  = torch.zeros(max_len, embed_dim)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, embed_dim, 2).float()
                        * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class Time2Vec(nn.Module):
    """
    Learnable time encoding (Kazemi et al., 2019).
      t2v(t) = [w0*t + b0,  sin(w1*t + b1), ..., sin(w_{D-1}*t + b_{D-1})]
    One linear trend term + (embed_dim - 1) learnable sinusoids.
    Input : scalar time feature  [B, T]
    Output: time embedding        [B, T, embed_dim]
    """
    def __init__(self, embed_dim: int):
        super().__init__()
        assert embed_dim >= 2, 'Time2Vec needs embed_dim >= 2'
        self.linear   = nn.Linear(1, 1)               # trend term
        self.periodic = nn.Linear(1, embed_dim - 1)   # periodic terms

    def forward(self, x):                             # x: [B, T]
        x   = x.unsqueeze(-1)                         # [B, T, 1]
        lin = self.linear(x)                          # [B, T, 1]
        per = torch.sin(self.periodic(x))             # [B, T, D-1]
        return torch.cat([lin, per], dim=-1)          # [B, T, D]


class DayEmbeddingTransformer(nn.Module):
    def __init__(self, vocab_size, pad_type_id, embed_dim=32,
                 nhead=4, num_layers=2, hidden_dim=128, dropout=0.2):
        super().__init__()
        assert embed_dim % nhead == 0
        self.pad_type_id = pad_type_id

        # Token feature projections
        self.type_emb  = nn.Embedding(vocab_size, embed_dim,
                                       padding_idx=pad_type_id)
        self.value_fc  = nn.Linear(1, embed_dim)

        # Time2Vec encodings (replaces plain linear hour_fc)
        self.hour_t2v  = Time2Vec(embed_dim)   # within-day time
        self.tfs_t2v   = Time2Vec(embed_dim)   # ← NEW: time from patient start

        self.input_norm = nn.LayerNorm(embed_dim)
        self.input_drop = nn.Dropout(dropout)
        self.pos_enc    = PositionalEncoding(embed_dim, max_len=30000)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead,
            dim_feedforward=hidden_dim,
            dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.embed_proj = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim),
            nn.Tanh())

        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, test_type, test_value, hour,
                time_from_start, padding_mask):      # ← time_from_start added
        x = (self.type_emb(test_type)               # [B, T, D]
             + self.value_fc(test_value.unsqueeze(-1))
             + self.hour_t2v(hour)                  # Time2Vec for hour
             + self.tfs_t2v(time_from_start))       # ← NEW: Time2Vec for tfs
        x = self.input_drop(self.input_norm(x))
        x = self.pos_enc(x)
        x = self.encoder(x, src_key_padding_mask=padding_mask)

        valid  = (~padding_mask).unsqueeze(-1).float()
        pooled = (x * valid).sum(1) / valid.sum(1).clamp(min=1)

        embedding = self.embed_proj(pooled)
        logit     = self.classifier(embedding)
        return embedding, logit


print('Model classes defined: PositionalEncoding, Time2Vec, DayEmbeddingTransformer')


Model classes defined: PositionalEncoding, Time2Vec, DayEmbeddingTransformer


---
## Step 7 — Training

BCE loss with pos_weight for class imbalance.
Mixed precision (float16) to reduce GPU memory.
Monitors val AUROC — saves best model checkpoint.


In [ ]:
BASE_SAVE = "/content/drive/My Drive/Monmouth_University/2025_RA/Medical_AI/Best_Model"

DIM_CONFIGS = {
    # 32:  {'nhead': 4, 'hidden_dim': 128},
    # 64:  {'nhead': 4, 'hidden_dim': 256},
    128: {'nhead': 4, 'hidden_dim': 512},
}

# ── Functions defined OUTSIDE the loop ───────────────────────────────────────
def run_epoch(loader, model, optimizer, criterion, scaler, train=True):
    model.train() if train else model.eval()
    total_loss, all_logits, all_labels = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    desc = 'Train' if train else 'Val  '
    with ctx:
        pbar = tqdm(loader, desc=f'    {desc}', leave=False,
                    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')
        for batch in pbar:
            tt  = batch['test_type'].to(device)
            tv  = batch['test_value'].to(device)
            ho  = batch['hour'].to(device)
            tfs = batch['time_from_start'].to(device)
            pm  = batch['padding_mask'].to(device)
            lbl = batch['label'].to(device)
            with autocast():
                _, logit = model(tt, tv, ho, tfs, pm)
                loss = criterion(logit.squeeze(1), lbl)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            total_loss += loss.item() * len(lbl)
            all_logits.append(logit.squeeze(1).detach().float().cpu())
            all_labels.append(lbl.cpu())
    all_logits = torch.cat(all_logits).sigmoid().numpy()
    all_labels = torch.cat(all_labels).numpy()
    avg_loss   = total_loss / len(all_labels)
    try:    auroc = roc_auc_score(all_labels, all_logits)
    except: auroc = float('nan')
    return avg_loss, auroc


def extract_embeddings(loader, model, device):
    accum = defaultdict(lambda: {'embs': [], 'label': None})
    with torch.no_grad():
        for batch in loader:
            tt  = batch['test_type'].to(device)
            tv  = batch['test_value'].to(device)
            ho  = batch['hour'].to(device)
            tfs = batch['time_from_start'].to(device)
            pm  = batch['padding_mask'].to(device)
            with autocast():
                emb, _ = model(tt, tv, ho, tfs, pm)
            emb_np = emb.float().cpu().numpy()
            for j in range(len(batch['label'])):
                key = (batch['patient_id'][j], batch['unified_day'][j])
                accum[key]['embs'].append(emb_np[j])
                accum[key]['label'] = int(batch['label'][j].item())
    records = []
    for (pid, uday), val in accum.items():
        mean_emb = np.mean(val['embs'], axis=0)
        row = {'patient_id': pid, 'unified_day': uday, 'label': val['label']}
        for k, v in enumerate(mean_emb):
            row[f'emb_{k}'] = float(v)
        records.append(row)
    return (pd.DataFrame(records)
            .sort_values(['patient_id', 'unified_day'])
            .reset_index(drop=True))


# ── Main loop ─────────────────────────────────────────────────────────────────
all_dim_results = {}

for EMBED_DIM, cfg in DIM_CONFIGS.items():
    print('\n' + '='*70)
    print(f'  EMBED_DIM = {EMBED_DIM}  |  nhead={cfg["nhead"]}  |  hidden={cfg["hidden_dim"]}')
    print('='*70)

    SAVE_DIR        = os.path.join(BASE_SAVE,
                          f'daily_embedding_classifier_last7days_{EMBED_DIM}dim_withtime')
    os.makedirs(SAVE_DIR, exist_ok=True)
    BEST_MODEL_PATH = os.path.join(SAVE_DIR, 'best_day_embedding_transformer.pth')
    print(f'  Save dir: {SAVE_DIR}')

    model = DayEmbeddingTransformer(
        vocab_size=VOCAB_SIZE, pad_type_id=PAD_TYPE_ID,
        embed_dim=EMBED_DIM, nhead=cfg['nhead'],
        num_layers=2, hidden_dim=cfg['hidden_dim'], dropout=0.2,
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Parameters: {n_params:,}')

    train_labels = [s[2] for s in train_ds.samples]
    n_pos = sum(train_labels); n_neg = len(train_labels) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=10)
    scaler = GradScaler()

    NUM_EPOCHS = 100; EARLY_STOP_PAT = 10
    best_val_auroc = 0.0; early_stop_cnt = 0
    train_losses, val_losses, train_aurocs, val_aurocs = [], [], [], []

    print(f'\n  {"Epoch":>6}  {"Train Loss":>10}  {"Train AUC":>9}  '
          f'{"Val Loss":>8}  {"Val AUC":>7}  {"Time":>7}')
    print('  ' + '-'*62)

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_auc = run_epoch(train_loader, model, optimizer,
                                    criterion, scaler, train=True)
        va_loss, va_auc = run_epoch(val_loader, model, optimizer,
                                    criterion, scaler, train=False)
        elapsed = time.time() - t0

        train_losses.append(tr_loss); val_losses.append(va_loss)
        train_aurocs.append(tr_auc);  val_aurocs.append(va_auc)
        scheduler.step(va_auc if not np.isnan(va_auc) else 0.0)

        marker = ''
        if (not np.isnan(va_auc)) and va_auc > best_val_auroc:
            best_val_auroc = va_auc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            early_stop_cnt = 0; marker = '✅'
        else:
            early_stop_cnt += 1
            if early_stop_cnt >= EARLY_STOP_PAT:
                print(f'  Early stopping at epoch {epoch}'); break

        va_str = f'{va_auc:.4f}' if not np.isnan(va_auc) else '   nan'
        print(f'  {epoch:>6}  {tr_loss:>10.4f}  {tr_auc:>9.4f}  '
              f'{va_loss:>8.4f}  {va_str:>7}  {elapsed:>6.1f}s  {marker}')

    print(f'\n  Best val AUROC (dim={EMBED_DIM}): {best_val_auroc:.4f}')

    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'embed_dim={EMBED_DIM} — Last 7 Days (w/ Time2Vec)', fontsize=12)
    axes[0].plot(train_losses, label='Train'); axes[0].plot(val_losses, label='Val')
    axes[0].set_title('BCE Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(train_aurocs, label='Train'); axes[1].plot(val_aurocs, label='Val')
    axes[1].set_title('AUROC'); axes[1].set_ylim(0, 1.05)
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
    plt.show()

    # Extract embeddings
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    model.eval()
    print(f'  Extracting embeddings (dim={EMBED_DIM})...')
    emb_train = extract_embeddings(train_loader, model, device)
    emb_val   = extract_embeddings(val_loader,   model, device)
    emb_test  = extract_embeddings(test_loader,  model, device)

    emb_train.to_parquet(os.path.join(SAVE_DIR, 'emb_train.parquet'), index=False)
    emb_val.to_parquet(  os.path.join(SAVE_DIR, 'emb_val.parquet'),   index=False)
    emb_test.to_parquet( os.path.join(SAVE_DIR, 'emb_test.parquet'),  index=False)
    print(f'  Embeddings saved → {SAVE_DIR}')
    print(f'  Train: {emb_train.shape}  Val: {emb_val.shape}  Test: {emb_test.shape}')

    all_dim_results[EMBED_DIM] = {
        'best_val_auroc': best_val_auroc,
        'save_dir': SAVE_DIR,
        'emb_train': emb_train, 'emb_val': emb_val, 'emb_test': emb_test,
    }
    del model; gc.collect(); torch.cuda.empty_cache()

print('\n' + '='*50)
print('FINAL SUMMARY — Best Val AUROC per embed_dim (w/ Time2Vec)')
print('='*50)
for dim, res in all_dim_results.items():
    print(f'  dim={dim:>3}: {res["best_val_auroc"]:.4f}  → {res["save_dir"]}')


In [ ]:
# Training curves now saved inside the dim loop above.
print('Training curves saved per dim inside the loop.')

---
## Step 8 — Extract Day Embeddings from Best Model

Load best checkpoint. For each (patient, unified_day):
- Run all 24k-row chunks through transformer
- Average chunk embeddings → one 32-dim vector per patient-day

Result: one row per (patient, unified_day) with columns emb_0..emb_31


In [ ]:
# Embeddings extracted inside the dim loop above.
# emb_train/val/test set to last dim by default from all_dim_results.
EMBED_DIM = list(all_dim_results.keys())[-1]
emb_train = all_dim_results[EMBED_DIM]['emb_train']
emb_val   = all_dim_results[EMBED_DIM]['emb_val']
emb_test  = all_dim_results[EMBED_DIM]['emb_test']
SAVE_DIR  = all_dim_results[EMBED_DIM]['save_dir']
print(f'Default dim={EMBED_DIM} set for downstream cells.')

In [ ]:
# Parquet already saved inside the dim loop.
print('Embeddings already saved per dim.')

In [9]:
# ── Extract 128-dim embeddings from saved checkpoint (NO RETRAINING) ──────────

import os, gc
import numpy as np
import pandas as pd
import torch
from collections import defaultdict
from torch.cuda.amp import autocast

# ── Config — match exactly what you trained with ──────────────────────────────
EMBED_DIM = 128
BASE_SAVE = "/content/drive/My Drive/Monmouth_University/2025_RA/Medical_AI/Best_Model"
SAVE_DIR  = os.path.join(BASE_SAVE, f'daily_embedding_classifier_last7days_{EMBED_DIM}dim_withtime')
BEST_MODEL_PATH = os.path.join(SAVE_DIR, 'best_day_embedding_transformer.pth')

print(f'Loading checkpoint from:\n  {BEST_MODEL_PATH}')
assert os.path.exists(BEST_MODEL_PATH), "❌ Checkpoint not found! Check the path."

# ── Rebuild model with same architecture ─────────────────────────────────────
model = DayEmbeddingTransformer(
    vocab_size=VOCAB_SIZE, pad_type_id=PAD_TYPE_ID,
    embed_dim=EMBED_DIM, nhead=4,
    num_layers=2, hidden_dim=512, dropout=0.2,
).to(device)

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
print(f'✅ Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')

# ── Extract embeddings ────────────────────────────────────────────────────────
def extract_embeddings(loader, model, device):
    accum = defaultdict(lambda: {'embs': [], 'label': None})
    with torch.no_grad():
        for batch in loader:
            tt  = batch['test_type'].to(device)
            tv  = batch['test_value'].to(device)
            ho  = batch['hour'].to(device)
            tfs = batch['time_from_start'].to(device)
            pm  = batch['padding_mask'].to(device)
            with autocast():
                emb, _ = model(tt, tv, ho, tfs, pm)
            emb_np = emb.float().cpu().numpy()
            for j in range(len(batch['label'])):
                key = (batch['patient_id'][j], batch['unified_day'][j])
                accum[key]['embs'].append(emb_np[j])
                accum[key]['label'] = int(batch['label'][j].item())
    records = []
    for (pid, uday), val in accum.items():
        mean_emb = np.mean(val['embs'], axis=0)
        row = {'patient_id': pid, 'unified_day': uday, 'label': val['label']}
        for k, v in enumerate(mean_emb):
            row[f'emb_{k}'] = float(v)
        records.append(row)
    return (pd.DataFrame(records)
            .sort_values(['patient_id', 'unified_day'])
            .reset_index(drop=True))

print('Extracting train embeddings...')
emb_train = extract_embeddings(train_loader, model, device)
print('Extracting val embeddings...')
emb_val   = extract_embeddings(val_loader,   model, device)
print('Extracting test embeddings...')
emb_test  = extract_embeddings(test_loader,  model, device)

# ── Save to parquet ───────────────────────────────────────────────────────────
emb_train.to_parquet(os.path.join(SAVE_DIR, 'emb_train.parquet'), index=False)
emb_val.to_parquet(  os.path.join(SAVE_DIR, 'emb_val.parquet'),   index=False)
emb_test.to_parquet( os.path.join(SAVE_DIR, 'emb_test.parquet'),  index=False)

print(f'\n✅ Embeddings saved → {SAVE_DIR}')
print(f'   Train: {emb_train.shape}')
print(f'   Val:   {emb_val.shape}')
print(f'   Test:  {emb_test.shape}')

del model; gc.collect(); torch.cuda.empty_cache()

Loading checkpoint from:
  /content/drive/My Drive/Monmouth_University/2025_RA/Medical_AI/Best_Model/daily_embedding_classifier_last7days_128dim_withtime/best_day_embedding_transformer.pth
✅ Model loaded. Parameters: 414,849
Extracting train embeddings...
Extracting val embeddings...
Extracting test embeddings...

✅ Embeddings saved → /content/drive/My Drive/Monmouth_University/2025_RA/Medical_AI/Best_Model/daily_embedding_classifier_last7days_128dim_withtime
   Train: (325, 131)
   Val:   (100, 131)
   Test:  (151, 131)
